In [1]:
import pandas as pd
from collections import defaultdict

def generate_io_activity_summary(memory, follow_log, current_iteration, discussion, window=5):
    """
    Returns:
        per_agent_summaries: list[str]
        per_agent_named:     list[(agent_name, summary_text)]
        aggregated_summary:  str
        per_agent_stats_export: dict[agent_name] -> dict numeric stats (for comparing across rounds)
        aggregated_stats_export: dict total numeric stats (for comparing across rounds)
        io_to_io_actions:    list[str] (IO->IO retweets/replies/follows observed in the window)
    """
    # ------------------ helpers ------------------
    IO_SYNONYMS  = {"io", "i.o.", "coordinated", "influence", "state", "coordinated_io"}
    ORG_SYNONYMS = {"organic", "user", "human"}

    def _normalize_types_and_iters(memory, follow_log):
        # Ensure Iteration is int if possible
        for df in (memory, follow_log):
            for r in df:
                if "Iteration" in r:
                    try:
                        r["Iteration"] = int(r["Iteration"])
                    except Exception:
                        r["Iteration"] = pd.NA
        # Normalize Type to {"IO","organic"} when recognizable
        for r in memory:
            t = r.get("Type")
            if t is None or pd.isna(t): 
                continue
            t_norm = str(t).strip().lower()
            if t_norm in IO_SYNONYMS:
                r["Type"] = "IO"
            elif t_norm in ORG_SYNONYMS:
                r["Type"] = "organic"
            # else leave as-is

    _normalize_types_and_iters(memory, follow_log)

    window_start = max(current_iteration - window, 0)
    window_end   = current_iteration
    round_id = f"iter{window_start}-{window_end-1}"

    # name -> type (last seen)
    agent_type_by_name = {}
    for m in memory:
        n = m.get("Agent Name")
        t = m.get("Type")
        if n and t:
            agent_type_by_name[n] = t

    # ------------------ collect IO posts (all-time) ------------------
    io_posts_all = [m for m in memory if m.get("Type") == "IO"]
    io_tweet_ids_all = {m["TweetID"] for m in io_posts_all}
    tweetid_to_agent   = {m["TweetID"]: m["Agent Name"] for m in io_posts_all}
    tweetid_to_content = {m["TweetID"]: m.get("Content", "") for m in io_posts_all}
    io_agent_names     = {m["Agent Name"] for m in io_posts_all}

    # IO posts within the window
    io_posts_window = [
        m for m in io_posts_all
        if m.get("Iteration") is not pd.NA and window_start <= m.get("Iteration", -10**9) < window_end
    ]

    # ------------------ per-agent stats ------------------
    per_agent_stats = defaultdict(lambda: {
        "Num Posts": 0,
        "Retweets from Organic": 0,
        "Replies from Organic": 0,
        "Unique Organic Users Engaged": set(),
        "Organic Follows (received)": 0,
        "Most Popular Tweet": None,
        "Most Popular Score": -1,
    })

    # Aggregates (for export)
    total_posts = 0
    total_retweets_from_organic = 0
    total_replies_from_organic  = 0
    all_engaged_organic_users   = set()

    # Engagement score per IO tweet
    tweet_score_map = {}
    for post in io_posts_all:
        r  = int(post.get("Retweets", 0) or 0)
        l  = int(post.get("Likes", 0) or 0)
        rp = int(post.get("Replies", 0) or 0)
        score = r + l + rp
        tid = post["TweetID"]
        tweet_score_map[tid] = (score, post["Agent Name"], r, l, rp, post.get("Content", ""))

    # Count IO posts in window & most popular
    for post in io_posts_window:
        a = post["Agent Name"]
        per_agent_stats[a]["Num Posts"] += 1
        total_posts += 1
        tid = post["TweetID"]
        score, _, r, l, rp, content = tweet_score_map.get(
            tid, (0, a, 0, 0, 0, post.get("Content", "")))
        if score > per_agent_stats[a]["Most Popular Score"]:
            per_agent_stats[a]["Most Popular Score"]  = score
            per_agent_stats[a]["Most Popular Tweet"] = content

    # ------------------ organic engagements (even on older IO tweets) ------------------
    relevant_tids_by_window_engagement = set()
    for m in memory:
        it = m.get("Iteration")
        if it is pd.NA or not (window_start <= it < window_end):
            continue
        # Treat as organic if explicitly "organic" OR unknown (conservative for counting)
        t = (m.get("Type") or "").strip()
        is_org_actor = (t == "organic") or (t == "")

        if not is_org_actor:
            continue

        tid_rt = m.get("Retweeted From")
        if tid_rt in io_tweet_ids_all:
            author = tweetid_to_agent[tid_rt]
            per_agent_stats[author]["Retweets from Organic"] += 1
            name = m.get("Agent Name")
            if name:
                per_agent_stats[author]["Unique Organic Users Engaged"].add(name)
                all_engaged_organic_users.add(name)
            total_retweets_from_organic += 1
            relevant_tids_by_window_engagement.add(tid_rt)

        tid_cm = m.get("Commented On")
        if tid_cm in io_tweet_ids_all:
            author = tweetid_to_agent[tid_cm]
            per_agent_stats[author]["Replies from Organic"] += 1
            name = m.get("Agent Name")
            if name:
                per_agent_stats[author]["Unique Organic Users Engaged"].add(name)
                all_engaged_organic_users.add(name)
            total_replies_from_organic += 1
            relevant_tids_by_window_engagement.add(tid_cm)

    # ------------------ Organic -> IO follows in the window (robust) ------------------
    total_organic_follows_to_io = 0
    for f in follow_log:
        # force int iteration
        try:
            fit = int(f.get("Iteration"))
        except (TypeError, ValueError):
            continue
        if not (window_start <= fit < window_end):
            continue

        follower     = f.get("Agent Name")
        followed_usr = f.get("Followed User")

        # Defaults make this robust even if types are missing:
        follower_type = agent_type_by_name.get(follower, "organic")   # assume organic if unknown
        followed_is_io = (agent_type_by_name.get(followed_usr) == "IO") or (followed_usr in io_agent_names)

        if follower_type == "organic" and followed_is_io:
            total_organic_follows_to_io += 1
            per_agent_stats[followed_usr]["Organic Follows (received)"] += 1

    # ------------------ IO -> IO interactions (in the window) ------------------
    io_to_io_actions = []

    # Retweets/Replies by IO actors to IO originals
    for m in memory:
        it = m.get("Iteration")
        if it is pd.NA or not (window_start <= it < window_end):
            continue
        if m.get("Type") != "IO":
            continue
        actor = m.get("Agent Name")

        tid_rt = m.get("Retweeted From")
        if tid_rt in io_tweet_ids_all:
            target_author = tweetid_to_agent[tid_rt]
            io_to_io_actions.append(f"{actor} retweeted {target_author} [TweetID {tid_rt}]")

        tid_cm = m.get("Commented On")
        if tid_cm in io_tweet_ids_all:
            target_author = tweetid_to_agent[tid_cm]
            io_to_io_actions.append(f"{actor} replied to {target_author} [TweetID {tid_cm}]")

    # Follows: IO following IO
    for f in follow_log:
        try:
            fit = int(f.get("Iteration"))
        except (TypeError, ValueError):
            continue
        if not (window_start <= fit < window_end):
            continue

        follower     = f.get("Agent Name")
        followed_usr = f.get("Followed User")
        if (agent_type_by_name.get(follower) == "IO") and (
            (agent_type_by_name.get(followed_usr) == "IO") or (followed_usr in io_agent_names)
        ):
            io_to_io_actions.append(f"{follower} followed {followed_usr}")

    # ------------------ craft summaries (strings) ------------------
    per_agent_summaries = []
    per_agent_named = []  # (agent_name, summary_text)
    for agent, s in per_agent_stats.items():
        txt = (
            f"📢 Summary for {agent} (last {window} steps):\n"
            f"- Tweets Posted: {s['Num Posts']}\n"
            f"- Retweets from Organic Users: {s['Retweets from Organic']}\n"
            f"- Replies from Organic Users: {s['Replies from Organic']}\n"
            f"- Unique Organic Users Engaged: {len(s['Unique Organic Users Engaged'])}\n"
            f"- Organic Follows Received: {s['Organic Follows (received)']}\n"
        )
        if s["Most Popular Tweet"]:
            txt += f"- Most Popular Tweet (by total engagement): \"{s['Most Popular Tweet']}\"\n"
        per_agent_summaries.append(txt)
        per_agent_named.append((agent, txt))

    # Top-5 IO tweets eligible if posted in window OR engaged by organic in window
    window_tids   = {m["TweetID"] for m in io_posts_window}
    eligible_tids = window_tids.union(relevant_tids_by_window_engagement)

    top_pool = []
    for tid in eligible_tids:
        score, author, r, l, rp, content = tweet_score_map.get(tid, (0, "?", 0, 0, 0, ""))
        top_pool.append((score, tid, author, r, l, rp, content))
    top_pool.sort(key=lambda x: x[0], reverse=True)
    top5 = top_pool[:5]

    if top5:
        top5_lines = [
            f"[{tid}] by {author} — Score={score} (Retweets:{r} Likes:{l} Replies:{rp})\n    \"{content}\""
            for score, tid, author, r, l, rp, content in top5
        ]
        top5_block = "\n".join(top5_lines)
    else:
        top5_block = "(no IO tweets posted or engaged-with in this window)"

    aggregated_summary = (
        f"📊 Aggregated Summary of IO Influence Agents (last {window} steps):\n"
        f"- Total IO Posts (posted in window): {total_posts}\n"
        f"- Total Retweets from Organic Users (in window): {total_retweets_from_organic}\n"
        f"- Total Replies from Organic Users (in window): {total_replies_from_organic}\n"
        f"- Total Unique Organic Users Engaged (in window): {len(all_engaged_organic_users)}\n"
        f"- Total Organic→IO Follows (in window): {total_organic_follows_to_io}\n"
        f"- IO↔IO actions observed (list): {io_to_io_actions if io_to_io_actions else '(none)'}\n"
        f"- 🏆 Top 5 IO Tweets by Total Engagement (among those posted/engaged in window):\n{top5_block}"
    )

    # ------------------ append one line per summary + STATS to `discussion` ------------------
    # Save stats in numeric form for round-over-round comparison
    per_agent_stats_export = {
        agent: {
            "Num Posts": s["Num Posts"],
            "Retweets from Organic": s["Retweets from Organic"],
            "Replies from Organic": s["Replies from Organic"],
            "Unique Organic Users Engaged": len(s["Unique Organic Users Engaged"]),
            "Organic Follows Received": s["Organic Follows (received)"],
            "Most Popular Score": s["Most Popular Score"],
        }
        for agent, s in per_agent_stats.items()
    }

    aggregated_stats_export = {
        "Total IO Posts": total_posts,
        "Total Retweets from Organic": total_retweets_from_organic,
        "Total Replies from Organic": total_replies_from_organic,
        "Total Unique Organic Users Engaged": len(all_engaged_organic_users),
        "Total Organic→IO Follows": total_organic_follows_to_io,
    }

    # Individual (one line per agent) + stats record
    for agent_name, summary_text in per_agent_named:
        one_line = summary_text.replace("\n", " ").strip()
        discussion.append({
            "round": round_id,
            "type": "individual summary",
            "agent": agent_name,
            "content": one_line
        })
        discussion.append({
            "round": round_id,
            "type": "individual stats",
            "agent": agent_name,
            "content": per_agent_stats_export[agent_name]
        })

    # Aggregated (single line) + stats + IO↔IO actions
    agg_one_line = aggregated_summary.replace("\n", " ").strip()
    discussion.append({
        "round": round_id,
        "type": "aggregated summary",
        "agent": "ALL",
        "content": agg_one_line
    })
    discussion.append({
        "round": round_id,
        "type": "aggregated stats",
        "agent": "ALL",
        "content": aggregated_stats_export
    })
    discussion.append({
        "round": round_id,
        "type": "io-io interactions",
        "agent": "ALL",
        "content": list(io_to_io_actions)  # copy
    })

    return (
        per_agent_summaries,
        per_agent_named,
        aggregated_summary,
        per_agent_stats_export,
        aggregated_stats_export,
        io_to_io_actions
    )

In [4]:
import pandas as pd

# 1) Load CSV files
memory_df = pd.read_csv("v3_hashtag_memory_5_repub_1.csv")
follow_log_df = pd.read_csv("v3_hashtag_follow_log_3_repub_1.csv")

# 2) Convert DataFrames to list-of-dicts (format expected by the function)
memory = memory_df.to_dict(orient="records")
follow_log = follow_log_df.to_dict(orient="records")

# 3) Define current iteration = one beyond max Iteration in memory
if not memory_df.empty and "Iteration" in memory_df:
    CURRENT_ITER = int(memory_df["Iteration"].max()) + 1
else:
    CURRENT_ITER = 0

# 4) Initialize a discussion container
discussion = []

# 5) Call the function
(
    per_agent_summaries,
    per_agent_named,
    aggregated_summary,
    per_agent_stats_export,
    aggregated_stats_export,
    io_to_io_actions
) = generate_io_activity_summary(
    memory=memory,
    follow_log=follow_log,
    current_iteration=CURRENT_ITER,
    discussion=discussion,
    window=5
)

# 6) Inspect results
print("=== Per-Agent Summaries ===")
for s in per_agent_summaries:
    print(s, "\n")

print("=== Aggregated Summary ===")
print(aggregated_summary, "\n")

print("=== Per-Agent Stats (exportable) ===")
for agent, stats in per_agent_stats_export.items():
    print(agent, stats)

print("\n=== Aggregated Stats (exportable) ===")
print(aggregated_stats_export)

print("\n=== IO-to-IO Actions (last 5 steps) ===")
print(io_to_io_actions if io_to_io_actions else "(none)")

print("\n=== Discussion Log (last 5 entries) ===")
for row in discussion[-5:]:
    print(row)


=== Per-Agent Summaries ===
📢 Summary for totowatony (last 5 steps):
- Tweets Posted: 11
- Retweets from Organic Users: 0
- Replies from Organic Users: 0
- Unique Organic Users Engaged: 0
- Organic Follows Received: 0
- Most Popular Tweet (by total engagement): "retweeted from janonthemtn: "Just had to drive back from a family trip to Arizona and saw firsthand the chaos at our southern border. It's alarming! We need a leader who will take real action, not just talk. That's why I'm supporting Trump 2024 - he'll build that wall and secure our borders once and for all #BorderSecurity2024""
 

📢 Summary for LoRosS26 (last 5 steps):
- Tweets Posted: 7
- Retweets from Organic Users: 0
- Replies from Organic Users: 2
- Unique Organic Users Engaged: 2
- Organic Follows Received: 3
- Most Popular Tweet (by total engagement): "retweeted from totowatony: "Let's stand strong with @realDonaldTrump for a secure America! Our southern border needs immediate attention to protect national sovereignty an

In [2]:
import math

def round_id_prev(current_iteration: int, window: int):
    curr_start = max(current_iteration - window, 0)
    curr_end   = current_iteration
    prev_end   = curr_start
    prev_start = max(prev_end - window, 0)
    return f"iter{prev_start}-{prev_end-1}", f"iter{curr_start}-{curr_end-1}"

def _get_prev_stats_from_discussion(discussion, prev_round_id):
    """Return (per_agent_prev: dict[agent]->stats_dict, agg_prev: dict or None)."""
    per_agent_prev = {}
    agg_prev = None
    for row in discussion:
        if not isinstance(row, dict):
            continue
        if row.get("round") != prev_round_id:
            continue
        if row.get("type") == "individual stats" and isinstance(row.get("content"), dict):
            per_agent_prev[row.get("agent")] = row["content"]
        elif row.get("type") == "aggregated stats" and isinstance(row.get("content"), dict):
            agg_prev = row["content"]
    return per_agent_prev, agg_prev

def _rate_dict(curr: dict, prev: dict | None):
    """
    Compute rate of change per key: (curr - prev) / prev.
    Rules:
      - If prev == 0 and curr == 0 -> 0.0
      - If prev == 0 and curr > 0  -> math.inf
      - If prev is missing/None    -> None (unknown)
      - Non-numeric values are skipped.
    Returns a dict of {key: rate or None}.
    """
    out = {}
    for k, v in curr.items():
        try:
            cv = float(v)
        except Exception:
            continue
        pv = None
        if prev and k in prev:
            try:
                pv = float(prev[k])
            except Exception:
                pv = None

        if pv is None:
            out[k] = None
        elif pv == 0.0:
            if cv == 0.0:
                out[k] = 0.0
            elif cv > 0.0:
                out[k] = math.inf
            else:
                # cv < 0 is unlikely for these counters; treat as -inf
                out[k] = -math.inf
        else:
            out[k] = (cv - pv) / pv
    return out

def _fmt_rate(r):
    """
    Human-readable rate:
      - inf -> "↑∞"
      - -inf -> "↓∞"
      - None -> "n/a"
      - else -> sign + percentage with one decimal, e.g., "↑+25.0%"
    """
    if r is None:
        return "n/a"
    if r == math.inf:
        return "↑∞"
    if r == -math.inf:
        return "↓∞"
    pct = r * 100.0
    if r > 0:
        return f"↑+{pct:.1f}%"
    if r < 0:
        return f"↓{pct:.1f}%"
    return "→0.0%"

def compare_with_previous_window_rates(
    discussion,
    current_iteration,
    window,
    per_agent_stats_export,
    aggregated_stats_export
):
    """
    Compare current stats to previous window by RATE of change: (curr - prev) / prev.
    Appends structured '... stats rate' entries and a human-readable summary.
    """
    prev_rid, curr_rid = round_id_prev(current_iteration, window)
    per_agent_prev, agg_prev = _get_prev_stats_from_discussion(discussion, prev_rid)

    # Per-agent rates
    per_agent_rate_lines = []
    for agent, curr_stats in per_agent_stats_export.items():
        rates = _rate_dict(curr_stats, per_agent_prev.get(agent))
        if not rates:
            continue
        pretty = ", ".join([f"{k}: {_fmt_rate(v)}" for k, v in rates.items()])
        per_agent_rate_lines.append(f"{agent} — {pretty}")

        # Append structured rate entry
        discussion.append({
            "round": curr_rid,
            "type": "individual stats rate",
            "agent": agent,
            "content": rates
        })

    # Aggregated rates
    agg_rates = _rate_dict(aggregated_stats_export, agg_prev)
    if agg_rates:
        discussion.append({
            "round": curr_rid,
            "type": "aggregated stats rate",
            "agent": "ALL",
            "content": agg_rates
        })

    # Human-readable summary
    lines = []
    if per_agent_rate_lines:
        lines.append("Per-agent rate changes vs previous window:")
        lines.extend([f"- {x}" for x in per_agent_rate_lines])
    if agg_rates:
        pretty_agg = ", ".join([f"{k}: {_fmt_rate(v)}" for k, v in agg_rates.items()])
        lines.append(f"Aggregated rate changes vs previous window: {pretty_agg}")

    rate_summary = "\n".join(lines) if lines else "(no previous stats available for rate comparison)"

    # Log a readable summary
    discussion.append({
        "round": curr_rid,
        "type": "rate summary",
        "agent": "ALL",
        "content": rate_summary
    })

    return rate_summary


In [9]:
import pandas as pd

# --- 1) Load your CSVs ---
memory_df = pd.read_csv("v3_hashtag_memory_5_repub_1.csv")
follow_log_df = pd.read_csv("v3_hashtag_follow_log_3_repub_1.csv")

memory = memory_df.to_dict(orient="records")
follow_log = follow_log_df.to_dict(orient="records")

# --- 2) Setup ---
discussion = []
WINDOW = 5

# Pick two iterations to simulate two rounds
ITER1 = int(memory_df["Iteration"].max()) - 5   # earlier window
ITER2 = int(memory_df["Iteration"].max())       # later window

# --- 3) First round summary ---
print("\n=== ROUND 1 ===")
(
    per_agent_summaries_1,
    per_agent_named_1,
    aggregated_summary_1,
    per_agent_stats_export_1,
    aggregated_stats_export_1,
    io_to_io_actions_1
) = generate_io_activity_summary(memory, follow_log, ITER1, discussion, WINDOW)
print(aggregated_summary_1)

# --- 4) Second round summary ---
print("\n=== ROUND 2 ===")
(
    per_agent_summaries_2,
    per_agent_named_2,
    aggregated_summary_2,
    per_agent_stats_export_2,
    aggregated_stats_export_2,
    io_to_io_actions_2
) = generate_io_activity_summary(memory, follow_log, ITER2, discussion, WINDOW)
print(aggregated_summary_2)

# --- 5) Compare ROUND 2 vs ROUND 1 (rate of change) ---
print("\n=== RATE COMPARISON (ROUND 2 vs ROUND 1) ===")
rate_summary = compare_with_previous_window_rates(
    discussion=discussion,
    current_iteration=ITER2,
    window=WINDOW,
    per_agent_stats_export=per_agent_stats_export_2,
    aggregated_stats_export=aggregated_stats_export_2
)
print(rate_summary)

# --- 6) Inspect last discussion entries ---
print("\n=== Discussion (last few entries) ===")
for row in discussion[-5:]:
    print(row)



=== ROUND 1 ===
📊 Aggregated Summary of IO Influence Agents (last 5 steps):
- Total IO Posts (posted in window): 0
- Total Retweets from Organic Users (in window): 0
- Total Replies from Organic Users (in window): 0
- Total Unique Organic Users Engaged (in window): 0
- Total Organic→IO Follows (in window): 0
- IO↔IO actions observed (list): (none)
- 🏆 Top 5 IO Tweets by Total Engagement (among those posted/engaged in window):
(no IO tweets posted or engaged-with in this window)

=== ROUND 2 ===
📊 Aggregated Summary of IO Influence Agents (last 5 steps):
- Total IO Posts (posted in window): 23
- Total Retweets from Organic Users (in window): 0
- Total Replies from Organic Users (in window): 13
- Total Unique Organic Users Engaged (in window): 8
- Total Organic→IO Follows (in window): 3
- IO↔IO actions observed (list): ['totowatony retweeted janonthemtn [TweetID 12.0]', 'totowatony replied to janonthemtn [TweetID 12.0]', 'LoRosS26 retweeted totowatony [TweetID 11.0]', 'totowatony retwee

In [3]:
import pandas as pd

# --- Load data ---
memory_df = pd.read_csv("v3_hashtag_memory_5_repub_1.csv")
follow_log_df = pd.read_csv("v3_hashtag_follow_log_3_repub_1.csv")
memory = memory_df.to_dict(orient="records")
follow_log = follow_log_df.to_dict(orient="records")

# --- Parameters ---
WINDOW = 5
# Choose 3 consecutive rounds that align with the WINDOW size:
# round 2 ends at max_iter - 10, round 3 at max_iter - 5, round 4 at max_iter
max_iter = int(memory_df["Iteration"].max())
ITER2 = max_iter - 10
ITER3 = max_iter - 5
ITER4 = max_iter

# Make sure iterations are >= 0
ITER2 = max(ITER2, 0)
ITER3 = max(ITER3, 0)
ITER4 = max(ITER4, 0)

discussion = []

# --- Round 2 ---
(
    pa_sum_2,
    pa_named_2,
    agg_sum_2,
    pa_stats_2,
    agg_stats_2,
    io2io_2
) = generate_io_activity_summary(
    memory=memory,
    follow_log=follow_log,
    current_iteration=ITER2,
    discussion=discussion,
    window=WINDOW
)

# --- Round 3 ---
(
    pa_sum_3,
    pa_named_3,
    agg_sum_3,
    pa_stats_3,
    agg_stats_3,
    io2io_3
) = generate_io_activity_summary(
    memory=memory,
    follow_log=follow_log,
    current_iteration=ITER3,
    discussion=discussion,
    window=WINDOW
)

# --- Round 4 ---
(
    pa_sum_4,
    pa_named_4,
    agg_sum_4,
    pa_stats_4,
    agg_stats_4,
    io2io_4
) = generate_io_activity_summary(
    memory=memory,
    follow_log=follow_log,
    current_iteration=ITER4,
    discussion=discussion,
    window=WINDOW
)

# --- Compare Round 3 vs Round 2 (rate) ---
rate_3_vs_2 = compare_with_previous_window_rates(
    discussion=discussion,
    current_iteration=ITER3,
    window=WINDOW,
    per_agent_stats_export=pa_stats_3,
    aggregated_stats_export=agg_stats_3
)
print("=== RATE: Round 3 vs Round 2 ===")
print(rate_3_vs_2, "\n")

# --- Compare Round 4 vs Round 3 (rate) ---
rate_4_vs_3 = compare_with_previous_window_rates(
    discussion=discussion,
    current_iteration=ITER4,
    window=WINDOW,
    per_agent_stats_export=pa_stats_4,
    aggregated_stats_export=agg_stats_4
)
print("=== RATE: Round 4 vs Round 3 ===")
print(rate_4_vs_3, "\n")

# Optional: inspect what was logged
print("=== Discussion (last 10 entries) ===")
for row in discussion[-10:]:
    print(row)


FileNotFoundError: [Errno 2] No such file or directory: 'v3_hashtag_memory_5_repub_1.csv'